# ADS-B multi-session specific-transmitter identification

This notebook combines every `.npy` capture in `data/` and uses the decoded ICAO address only to make the target label.

Rules:

- Models receive only samples 0--79 of the fixed ADS-B preamble.
- The payload begins at sample 80 and is never a model input.
- Source files are treated as capture sessions; repeated payloads and nearby same-aircraft records stay in one fold.
- In the two-class configuration, only the two most populous eligible aircraft are available for fitting. Every other aircraft is an `UNKNOWN` validation target, rejected with a threshold calibrated only on held-out known-class training groups.

Models: standardized physical-feature RBF SVM, physical-feature random forest, compact 1D CNN, compact TCN, and compact bidirectional GRU. Image-oriented MobileNet models were removed because a $2\times80$ I/Q preamble and this dataset size are a poor match for that architecture and its ImageNet pretraining.


In [1]:
# Run only if packages are missing.
# %pip install numpy pandas scikit-learn matplotlib torch "pyModeS<3"


In [2]:
# Configuration. This is the only source-code difference between the two notebooks.
from pathlib import Path

CONFIG = {
    "capture_dir": Path("data"),
    "capture_pattern": "*.npy",
    "sample_rate_hz": 10e6,
    "samples_per_half_symbol": 5,
    "preamble_half_symbols": 16,
    "message_bits": 112,
    "record_len": 1200,
    "preamble_len": 80,
    "min_records_per_class": 6,
    "known_class_count": 2,
    "max_records_per_class": None,
    "temporal_group_gap_seconds": 5.0,
    "requested_folds": 5,
    "random_seed": 42,
    "cnn_channels": (16, 32, 48),
    "gru_hidden": 32,
    "tcn_channels": 24,
    "dropout": 0.30,
    "batch_size": 64,
    "epochs": 15,
    "learning_rate": 1e-3,
    "weight_decay": 2e-4,
    "random_forest_trees": 200,
    "calibration_fraction": 0.20,
    "known_acceptance_target": 0.95,
    "experiments": (
        "physical_feature_svm",
        "physical_feature_random_forest",
        "preamble_cnn",
        "preamble_tcn",
        "preamble_gru_rnn",
    ),
}

assert CONFIG["preamble_len"] == CONFIG["preamble_half_symbols"] * CONFIG["samples_per_half_symbol"]
IS_OPEN_SET = CONFIG["known_class_count"] is not None
RUN_SLUG = "two_class_open_set" if IS_OPEN_SET else "all_eligible_classes"
CONFIG["artifact_dir"] = Path(
    "adsb_specific_transmitter_two_class_outputs"
    if IS_OPEN_SET else "adsb_specific_transmitter_raw_outputs"
)
CONFIG["figure_dir"] = Path("notes/figures") / f"20260921_{RUN_SLUG}"
CONFIG["artifact_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["figure_dir"].mkdir(parents=True, exist_ok=True)
CONFIG


{'capture_dir': PosixPath('data'),
 'capture_pattern': '*.npy',
 'sample_rate_hz': 10000000.0,
 'samples_per_half_symbol': 5,
 'preamble_half_symbols': 16,
 'message_bits': 112,
 'record_len': 1200,
 'preamble_len': 80,
 'min_records_per_class': 6,
 'known_class_count': 2,
 'max_records_per_class': None,
 'temporal_group_gap_seconds': 5.0,
 'requested_folds': 5,
 'random_seed': 42,
 'cnn_channels': (16, 32, 48),
 'gru_hidden': 32,
 'tcn_channels': 24,
 'dropout': 0.3,
 'batch_size': 64,
 'epochs': 15,
 'learning_rate': 0.001,
 'weight_decay': 0.0002,
 'random_forest_trees': 200,
 'calibration_fraction': 0.2,
 'known_acceptance_target': 0.95,
 'experiments': ('physical_feature_svm',
  'physical_feature_random_forest',
  'preamble_cnn',
  'preamble_tcn',
  'preamble_gru_rnn'),
 'artifact_dir': PosixPath('adsb_specific_transmitter_two_class_outputs'),
 'figure_dir': PosixPath('notes/figures/20260921_two_class_open_set')}

## Controls

- Decode the full record only for CRC validation, the ICAO label and a payload hash.
- Build every model input on a separate path from `iq[:80]`.
- Preserve the raw I/Q tensor exactly; fit SVM feature scaling and neural I/Q channel scaling inside each training fold only.
- Fit all model weights and open-set thresholds without using the outer validation fold.
- Keep duplicate payloads and nearby same-aircraft records in the same group, with temporal grouping performed separately inside each capture file.


In [3]:
# Imports and repeatability
import hashlib
import json
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyModeS as pms

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

if not hasattr(pms, "crc") or not hasattr(pms, "adsb"):
    raise ImportError('Install "pyModeS<3" for the decoder API used here.')

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["random_seed"])


Device: mps


In [4]:
# Load every appended timestamp/IQ capture file written by SaveQueue.

def resolve_capture_files(capture_dir, pattern):
    candidate_dirs = [
        Path(capture_dir),
        Path("../") / capture_dir,
        Path("../../") / capture_dir,
        Path("../../../") / capture_dir,
    ]
    for candidate_dir in candidate_dirs:
        files = sorted(path.resolve() for path in candidate_dir.glob(pattern) if path.is_file())
        if files:
            return files
    checked = "\n".join(str(path.resolve() / pattern) for path in candidate_dirs)
    raise FileNotFoundError(f"No capture files matched. Checked:\n{checked}")


def load_appended_npy_capture(path, first_global_index):
    rows = []
    file_size = path.stat().st_size

    with path.open("rb") as stream:
        source_record_index = 0
        while stream.tell() < file_size:
            pair_offset = stream.tell()
            try:
                timestamp_array = np.load(stream, allow_pickle=True)
                iq = np.load(stream, allow_pickle=False)
            except (EOFError, OSError, ValueError) as exc:
                raise ValueError(
                    f"Could not read pair {source_record_index} at byte offset {pair_offset} in {path}"
                ) from exc

            timestamp = timestamp_array.item() if timestamp_array.shape == () else timestamp_array.tolist()
            iq = np.asarray(iq)
            if iq.ndim != 1 or not np.iscomplexobj(iq):
                raise ValueError(
                    f"{path.name} record {source_record_index} is not one-dimensional complex I/Q: "
                    f"shape={iq.shape}, dtype={iq.dtype}"
                )

            rows.append({
                "record_index": first_global_index + source_record_index,
                "source_record_index": source_record_index,
                "capture_file": path.name,
                "session_id": path.stem,
                "timestamp": pd.Timestamp(timestamp),
                "sample_count": len(iq),
                "iq": iq.astype(np.complex64, copy=False),
            })
            source_record_index += 1

    return pd.DataFrame(rows)


capture_files = resolve_capture_files(CONFIG["capture_dir"], CONFIG["capture_pattern"])
capture_frames = []
next_global_index = 0
for capture_path in capture_files:
    capture_frame = load_appended_npy_capture(capture_path, next_global_index)
    capture_frames.append(capture_frame)
    next_global_index += len(capture_frame)

capture_records = pd.concat(capture_frames, ignore_index=True)
capture_file_audit = (
    capture_records.groupby(["capture_file", "session_id"], as_index=False)
    .agg(records=("record_index", "size"), capture_start=("timestamp", "min"), capture_end=("timestamp", "max"))
)
print(f"Loaded {len(capture_records):,} records from {len(capture_files)} files")
display(capture_file_audit)
print("Sample counts:", capture_records["sample_count"].value_counts().to_dict())


Loaded 1,556 records from 2 files


,capture_file,session_id,records,capture_start,capture_end
0,050926.npy,050926,558,2026-08-25 16:30:54.108478,2026-08-25 18:32:37.533011
1,150926.npy,150926,998,2026-08-25 20:32:26.372398,2026-08-26 01:40:45.997423


Sample counts: {1200: 1556}


In [5]:
# Decode only far enough to create the class label and split-audit hash.
# No decoded message or aircraft-state field is retained.

def demodulate_message(iq, config=CONFIG):
    iq = np.asarray(iq).reshape(-1)
    if len(iq) != config["record_len"]:
        raise ValueError(f"Expected {config['record_len']} samples, got {len(iq)}")

    half_symbols = np.abs(iq).reshape(-1, config["samples_per_half_symbol"]).mean(axis=1)
    payload = half_symbols[config["preamble_half_symbols"]:]
    bits = payload[0::2] > payload[1::2]
    if len(bits) != config["message_bits"]:
        raise ValueError(f"Expected {config['message_bits']} bits, got {len(bits)}")
    packed = np.packbits(bits).tobytes()
    return packed.hex().upper(), hashlib.sha256(packed).hexdigest()


def decode_labels(capture_df):
    decoded_rows = []
    for row in capture_df.itertuples(index=False):
        result = {
            "record_index": row.record_index,
            "icao": None,
            "payload_hash": None,
            "decode_ok": False,
            "decode_error": None,
        }
        try:
            message, payload_hash = demodulate_message(row.iq)
            if int(pms.crc(message)) != 0:
                result["decode_error"] = "CRC"
            else:
                result["icao"] = str(pms.adsb.icao(message)).upper()
                result["payload_hash"] = payload_hash
                result["decode_ok"] = True
        except Exception as exc:
            result["decode_error"] = f"{type(exc).__name__}: {exc}"
        decoded_rows.append(result)
    return pd.DataFrame(decoded_rows)


decoded_labels = decode_labels(capture_records)
records = capture_records.merge(decoded_labels, on="record_index", how="left", validate="one_to_one")

print("CRC-valid labelled records:", int(records["decode_ok"].sum()), "/", len(records))
display(records.drop(columns="iq").head())


CRC-valid labelled records: 1556 / 1556


,record_index,source_record_index,capture_file,session_id,timestamp,sample_count,icao,payload_hash,decode_ok,decode_error
0,0,0,050926.npy,050926,2026-08-25 16:30:54.108478,1200,7C3FCB,17c7ed73d35538a50b4bbde67660db4592609a3d11f05d...,True,None
1,1,1,050926.npy,050926,2026-08-25 16:30:57.178585,1200,7C3FCB,39e88429f3f1ffb145149ddd32d6df632efde3db2b587c...,True,None
2,2,2,050926.npy,050926,2026-08-25 16:31:02.613508,1200,7C3FCB,4a57ed1ef833b778ec5eff4118ddaea29d7bb70d8b4942...,True,None
3,3,3,050926.npy,050926,2026-08-25 16:31:03.621547,1200,7C3FCB,99e796907f0ce9d84acf6f0ef1c9a4d60d6f49e8087a0d...,True,None
4,4,4,050926.npy,050926,2026-08-25 16:31:16.110987,1200,7C3FCB,99e796907f0ce9d84acf6f0ef1c9a4d60d6f49e8087a0d...,True,None


In [6]:
# Dataset audit before model preparation.
valid_records = records.loc[records["decode_ok"]].copy()
valid_records["icao"] = valid_records["icao"].astype(str)

class_counts = valid_records["icao"].value_counts()
class_session_counts = pd.crosstab(valid_records["icao"], valid_records["capture_file"])
class_session_summary = valid_records.groupby("icao").agg(
    records=("record_index", "size"),
    sessions=("session_id", "nunique"),
    unique_payloads=("payload_hash", "nunique"),
).sort_values(["sessions", "records"], ascending=False)

print("Capture start:", valid_records["timestamp"].min())
print("Capture end:  ", valid_records["timestamp"].max())
print("ICAO classes:", valid_records["icao"].nunique())
print("Classes seen in multiple capture files:", int((class_session_summary["sessions"] > 1).sum()))
print("Unique payloads:", valid_records["payload_hash"].nunique())
print("Repeated records:", len(valid_records) - valid_records["payload_hash"].nunique())
display(class_session_summary)

plt.figure(figsize=(12, 4))
class_counts.plot(kind="bar")
plt.title("CRC-valid records per ICAO class across all capture files")
plt.xlabel("ICAO class (label only)")
plt.ylabel("Records")
plt.tight_layout()
plt.show()


Capture start: 2026-08-25 16:30:54.108478
Capture end:   2026-08-26 01:40:45.997423
ICAO classes: 62
Classes seen in multiple capture files: 3
Unique payloads: 1355
Repeated records: 201


,records,sessions,unique_payloads
icao,,,
7C7701,24,2,24
7C6C9B,23,2,23
7C7FE6,14,2,14
7C0B96,246,1,193
7C802A,64,1,60
...,...,...,...
7C487C,5,1,5
7C6B3D,5,1,5
7C6D94,5,1,5


/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_2112/1029197199.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Payload-free input

At 10 MSPS the fixed 8 microsecond preamble is samples 0--79. Pulse windows start at samples 0, 10, 35 and 45. The payload begins at sample 80.


In [7]:
# Build the content-free preamble tensor and engineered physical features.
PREAMBLE_PULSE_START_US = np.array([0.0, 1.0, 3.5, 4.5])
PREAMBLE_HALVES = np.zeros(CONFIG["preamble_half_symbols"], dtype=bool)
PREAMBLE_HALVES[(PREAMBLE_PULSE_START_US / 0.5).astype(int)] = True
PREAMBLE_ACTIVE_MASK = np.repeat(PREAMBLE_HALVES, CONFIG["samples_per_half_symbol"])
ACTIVE_HALF_INDICES = np.flatnonzero(PREAMBLE_HALVES)
assert ACTIVE_HALF_INDICES.tolist() == [0, 2, 7, 9]


def slice_raw_preamble(iq, config=CONFIG):
    preamble = np.asarray(iq[:config["preamble_len"]], dtype=np.complex64).copy()
    if len(preamble) != config["preamble_len"]:
        raise ValueError("Record is too short for the fixed ADS-B preamble")
    return preamble


def moment_features(values):
    values = np.asarray(values, dtype=np.float64)
    mean = values.mean()
    std = values.std() + 1e-12
    z = (values - mean) / std
    return float(mean), float(std), float(np.mean(z ** 3)), float(np.mean(z ** 4))


def circular_mean(values):
    return float(np.angle(np.mean(np.exp(1j * values))))


def extract_preamble_features(preamble, config=CONFIG):
    active = preamble[PREAMBLE_ACTIVE_MASK]
    quiet = preamble[~PREAMBLE_ACTIVE_MASK]
    pulse_matrix = np.stack([
        preamble[
            half_index * config["samples_per_half_symbol"]:
            (half_index + 1) * config["samples_per_half_symbol"]
        ]
        for half_index in ACTIVE_HALF_INDICES
    ])

    active_amp = np.abs(active)
    quiet_amp = np.abs(quiet)
    amp_mean, amp_std, amp_skew, amp_kurtosis = moment_features(active_amp)

    phase_steps = np.angle(pulse_matrix[:, 1:] * np.conj(pulse_matrix[:, :-1])).ravel()
    mean_phase_step = circular_mean(phase_steps)
    centred_steps = np.angle(np.exp(1j * (phase_steps - mean_phase_step)))
    cfo_hz = mean_phase_step * config["sample_rate_hz"] / (2 * np.pi)

    i_power = float(np.mean(active.real ** 2)) + 1e-12
    q_power = float(np.mean(active.imag ** 2)) + 1e-12
    iq_corr = float(np.corrcoef(active.real, active.imag)[0, 1])
    if not np.isfinite(iq_corr):
        iq_corr = 0.0

    pulse_prototype = pulse_matrix.mean(axis=0)
    pulse_amplitude = np.abs(pulse_matrix).mean(axis=0)
    pulse_phase = np.angle(pulse_prototype)
    per_pulse_power = np.mean(np.abs(pulse_matrix) ** 2, axis=1)

    feature_values = [
        amp_mean,
        amp_std,
        amp_skew,
        amp_kurtosis,
        float(active_amp.max() / (np.sqrt(np.mean(active_amp ** 2)) + 1e-12)),
        float(np.mean(active_amp ** 2)),
        float(np.mean(quiet_amp ** 2)),
        float(10 * np.log10((np.mean(active_amp ** 2) + 1e-12) /
                            (np.mean(quiet_amp ** 2) + 1e-12))),
        cfo_hz,
        float(np.std(centred_steps)),
        float(10 * np.log10(i_power / q_power)),
        iq_corr,
        float(np.std(per_pulse_power) / (np.mean(per_pulse_power) + 1e-12)),
    ]
    feature_values.extend(pulse_amplitude.tolist())
    feature_values.extend(pulse_prototype.real.tolist())
    feature_values.extend(pulse_prototype.imag.tolist())
    feature_values.extend(pulse_phase.tolist())
    return np.asarray(feature_values, dtype=np.float32)


FEATURE_NAMES = [
    "active_amplitude_mean", "active_amplitude_std", "active_amplitude_skew",
    "active_amplitude_kurtosis", "crest_factor", "active_power", "quiet_power",
    "preamble_snr_db", "carrier_offset_hz", "phase_step_std",
    "iq_power_imbalance_db", "iq_correlation", "pulse_power_variation",
]
FEATURE_NAMES += [f"pulse_amplitude_{i}" for i in range(5)]
FEATURE_NAMES += [f"pulse_i_{i}" for i in range(5)]
FEATURE_NAMES += [f"pulse_q_{i}" for i in range(5)]
FEATURE_NAMES += [f"pulse_phase_{i}" for i in range(5)]


def iq_to_channels(iq):
    return np.stack([iq.real, iq.imag], axis=0).astype(np.float32)


def prepare_model_table(records_df, config=CONFIG):
    valid = records_df.loc[records_df["decode_ok"]].copy()
    counts = valid["icao"].value_counts()
    eligible_counts = counts[counts >= config["min_records_per_class"]]

    if config["known_class_count"] is None:
        known_classes = eligible_counts.index.tolist()
        df = valid[valid["icao"].isin(known_classes)].copy()
    else:
        if len(eligible_counts) < config["known_class_count"]:
            raise ValueError(
                f"Need at least {config['known_class_count']} eligible classes; found {len(eligible_counts)}"
            )
        known_classes = eligible_counts.head(config["known_class_count"]).index.tolist()
        # All other valid aircraft remain in the evaluation set as UNKNOWN.
        df = valid.copy()

    if config["max_records_per_class"] is not None:
        known = (
            df[df["icao"].isin(known_classes)]
            .sample(frac=1, random_state=config["random_seed"])
            .groupby("icao", sort=False, group_keys=False)
            .head(config["max_records_per_class"])
        )
        unknown = df[~df["icao"].isin(known_classes)]
        df = pd.concat([known, unknown], ignore_index=False).sort_index()

    preambles = []
    features = []
    for iq in df["iq"]:
        preamble = slice_raw_preamble(iq, config)
        preambles.append(iq_to_channels(preamble))
        features.append(extract_preamble_features(preamble, config))

    model_metadata = df[[
        "record_index", "source_record_index", "capture_file", "session_id",
        "timestamp", "icao", "payload_hash",
    ]].reset_index(drop=True)
    model_metadata["is_known_class"] = model_metadata["icao"].isin(known_classes)
    model_metadata["evaluation_label"] = np.where(
        model_metadata["is_known_class"], model_metadata["icao"], "UNKNOWN"
    )
    return np.stack(preambles), np.stack(features), model_metadata, known_classes


X_preamble, X_physical, model_metadata, class_names = prepare_model_table(records, CONFIG)
class_to_index = {label: index for index, label in enumerate(class_names)}
y_model = np.array([class_to_index.get(label, -1) for label in model_metadata["icao"]], dtype=int)
y_evaluation = model_metadata["evaluation_label"].to_numpy()
evaluation_class_names = class_names + (["UNKNOWN"] if IS_OPEN_SET else [])

assert X_preamble.shape[1:] == (2, CONFIG["preamble_len"])
assert X_preamble.shape[-1] < CONFIG["record_len"]
assert X_physical.shape[1] == len(FEATURE_NAMES)
assert np.all(y_model[model_metadata["is_known_class"]] >= 0)
assert np.all(y_model[~model_metadata["is_known_class"]] == -1)

print("Raw payload-free I/Q tensor:", X_preamble.shape)
print("Physical feature matrix:", X_physical.shape)
print("Known training classes (population order):")
display(model_metadata.loc[model_metadata["is_known_class"], "icao"].value_counts().rename("records").to_frame())
print("Evaluation targets:")
display(model_metadata["evaluation_label"].value_counts().rename("records").to_frame())

# Confirm that every model tensor contains the original sample values.
iq_by_record = records.set_index("record_index")["iq"]
source_preambles = np.stack([
    iq_by_record.loc[index][:CONFIG["preamble_len"]] for index in model_metadata["record_index"]
])
assert np.array_equal(X_preamble[:, 0], source_preambles.real.astype(np.float32))
assert np.array_equal(X_preamble[:, 1], source_preambles.imag.astype(np.float32))
print("All raw sample checks passed")


Raw payload-free I/Q tensor: (1556, 2, 80)
Physical feature matrix: (1556, 33)
Known training classes (population order):


,records
icao,
7C0B96,246
7C802A,64


Evaluation targets:


,records
evaluation_label,
UNKNOWN,1246
7C0B96,246
7C802A,64


All raw sample checks passed


In [8]:
# Example raw input. It stops before the payload.
figure_dir = CONFIG["figure_dir"]
example_idx = int(np.flatnonzero(model_metadata["is_known_class"].to_numpy())[0])

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(X_preamble[example_idx, 0], label="I")
axes[0].plot(X_preamble[example_idx, 1], label="Q")
axes[0].set_ylabel("Raw I/Q")
axes[0].legend()
axes[1].plot(np.sqrt(np.sum(X_preamble[example_idx] ** 2, axis=0)), label="Magnitude")
for start in ACTIVE_HALF_INDICES * CONFIG["samples_per_half_symbol"]:
    axes[1].axvspan(start, start + CONFIG["samples_per_half_symbol"], color="0.9")
axes[1].set_xlabel("Preamble sample")
axes[1].set_ylabel("Magnitude")
axes[1].legend()
fig.suptitle(f"Raw payload-free model input - ICAO class {model_metadata.loc[example_idx, 'icao']}")
plt.tight_layout()
fig.savefig(figure_dir / "raw_preamble_input.png", dpi=180, bbox_inches="tight")
plt.show()


/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_2112/3721401352.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Raw I/Q check

The plots show unchanged I and Q values after slicing samples 0--79. They show the known training classes; unknown aircraft remain evaluation-only in the two-class configuration.


In [9]:
raw_preambles = X_preamble[:, 0] + 1j * X_preamble[:, 1]
q_fraction = np.mean(raw_preambles.imag ** 2, axis=1) / (
    np.mean(np.abs(raw_preambles) ** 2, axis=1) + 1e-12
)
iq_channel_summary = model_metadata.loc[model_metadata["is_known_class"], ["icao"]].copy()
iq_channel_summary["raw_q_energy_percent"] = 100 * q_fraction[model_metadata["is_known_class"]]
iq_channel_summary = iq_channel_summary.groupby("icao", sort=True).median().reset_index()
print(f"All-record median raw Q energy: {100 * np.median(q_fraction):.1f}%")
display(iq_channel_summary.round(1))

representative_indices = {}
icao_values = model_metadata["icao"].to_numpy()
for icao in class_names:
    indices = np.flatnonzero(icao_values == icao)
    magnitudes = np.abs(raw_preambles[indices])
    median_magnitude = np.median(magnitudes, axis=0)
    representative_indices[icao] = indices[
        np.argmin(np.mean((magnitudes - median_magnitude) ** 2, axis=1))
    ]

n_cols = min(3, len(class_names))
n_rows = int(np.ceil(len(class_names) / n_cols))
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(5 * n_cols, 2.7 * n_rows),
    sharex=True, sharey=True, squeeze=False,
)
pulse_starts = ACTIVE_HALF_INDICES * CONFIG["samples_per_half_symbol"]
display_limit = np.quantile(np.abs(np.concatenate([
    raw_preambles.real.ravel(), raw_preambles.imag.ravel()
])), 0.995)

for class_position, icao in enumerate(class_names):
    axis = axes.ravel()[class_position]
    values = raw_preambles[representative_indices[icao]]
    for start in pulse_starts:
        axis.axvspan(start, start + CONFIG["samples_per_half_symbol"], color="0.93", zorder=0)
    axis.plot(values.real, label="I", linewidth=0.9)
    axis.plot(values.imag, label="Q", linewidth=0.9)
    axis.axhline(0, color="0.75", linewidth=0.5)
    axis.set_title(icao, fontsize=9)
    axis.set_xlim(0, CONFIG["preamble_len"] - 1)
    axis.set_ylim(-display_limit, display_limit)
    axis.tick_params(labelsize=7)

for axis in axes.ravel()[len(class_names):]:
    axis.axis("off")
axes[0, 0].legend(fontsize=7, loc="upper right")
fig.supxlabel("Preamble sample; grey regions are the four known pulse windows")
fig.supylabel("Raw amplitude")
fig.suptitle("Representative raw preamble I/Q for known training classes", y=0.995)
plt.tight_layout(rect=(0.02, 0.02, 1, 0.985))
fig.savefig(figure_dir / "known_classes_raw_iq.png", dpi=160, bbox_inches="tight")
plt.show()


All-record median raw Q energy: 51.3%


,icao,raw_q_energy_percent
0,7C0B96,52.0
1,7C802A,54.5


/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_2112/4007205625.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Session-aware grouped folds

Each `.npy` file is a capture session. Exact repeated payloads for the same aircraft are connected even if they occur in different files. Otherwise, records are connected only when they belong to the same aircraft and capture file and are no more than five seconds apart. This prevents short bursts from straddling folds while still allowing validation when an aircraft appears in only one session.

This is not a strict held-out-session or held-out-day experiment: that design requires each evaluated aircraft to occur in enough independent sessions.


In [10]:
# Connected components for duplicate-payload and within-session temporal grouping.

def connected_groups(metadata, config=CONFIG):
    group_ids = np.empty(len(metadata), dtype=object)

    for icao, class_frame in metadata.groupby("icao", sort=True):
        positions = class_frame.index.to_numpy()
        local_index = {global_idx: i for i, global_idx in enumerate(positions)}
        parent = list(range(len(positions)))

        def find(i):
            while parent[i] != i:
                parent[i] = parent[parent[i]]
                i = parent[i]
            return i

        def union(a, b):
            root_a, root_b = find(a), find(b)
            if root_a != root_b:
                parent[root_b] = root_a

        for _, same_payload in class_frame.groupby("payload_hash"):
            members = same_payload.index.to_list()
            for member in members[1:]:
                union(local_index[members[0]], local_index[member])

        for _, session_frame in class_frame.groupby("session_id"):
            ordered = session_frame.sort_values("timestamp").index.to_list()
            for left, right in zip(ordered, ordered[1:]):
                delta = (metadata.loc[right, "timestamp"] - metadata.loc[left, "timestamp"]).total_seconds()
                if delta <= config["temporal_group_gap_seconds"]:
                    union(local_index[left], local_index[right])

        roots = {}
        for global_idx in positions:
            root = find(local_index[global_idx])
            roots.setdefault(root, len(roots))
            group_ids[global_idx] = f"{icao}:group_{roots[root]:03d}"

    return group_ids


groups = connected_groups(model_metadata, CONFIG)
model_metadata["group_id"] = groups

groups_per_original_class = model_metadata.groupby("icao")["group_id"].nunique()
groups_per_evaluation_class = model_metadata.groupby("evaluation_label")["group_id"].nunique()
display(pd.DataFrame({
    "records": model_metadata["icao"].value_counts(),
    "groups": groups_per_original_class,
    "sessions": model_metadata.groupby("icao")["session_id"].nunique(),
}).sort_values("records", ascending=False))

n_splits = min(CONFIG["requested_folds"], int(groups_per_evaluation_class.min()))
if len(class_names) < 2 or n_splits < 2:
    raise ValueError(
        "Grouped validation is not possible: every evaluation target needs at least two groups."
    )

evidence_grade = "session-aware temporal-burst grouped evaluation"
print("Validation:", evidence_grade)
print(f"Using {n_splits} folds (requested {CONFIG['requested_folds']}).")
print("Groups per evaluation target:", groups_per_evaluation_class.to_dict())


,records,groups,sessions
icao,,,
7C0B96,246,31,1
7C802A,64,7,1
AB0763,54,5,1
7C49D9,47,2,1
7C47A6,43,6,1
...,...,...,...
7C487C,5,5,1
7C6B3D,5,1,1
7C6D94,5,3,1


Validation: session-aware temporal-burst grouped evaluation
Using 5 folds (requested 5).
Groups per evaluation target: {'7C0B96': 31, '7C802A': 7, 'UNKNOWN': 286}


In [11]:
# Deterministic stratified assignment of groups to outer validation folds.

def make_stratified_group_folds(labels, group_ids, n_splits, seed):
    labels = np.asarray(labels)
    group_ids = np.asarray(group_ids)
    rng = np.random.default_rng(seed)
    fold_groups = [set() for _ in range(n_splits)]
    global_fold_load = np.zeros(n_splits, dtype=int)

    for label in np.unique(labels):
        class_mask = labels == label
        class_groups, counts = np.unique(group_ids[class_mask], return_counts=True)
        order = rng.permutation(len(class_groups))
        class_groups = class_groups[order]
        counts = counts[order]
        size_order = np.argsort(-counts, kind="stable")
        class_fold_load = np.zeros(n_splits, dtype=int)

        for group_id, group_size in zip(class_groups[size_order], counts[size_order]):
            least_class_load = class_fold_load.min()
            candidates = np.flatnonzero(class_fold_load == least_class_load)
            fold = int(candidates[np.argmin(global_fold_load[candidates])])
            fold_groups[fold].add(group_id)
            class_fold_load[fold] += int(group_size)
            global_fold_load[fold] += int(group_size)

    folds = []
    for validation_groups in fold_groups:
        val_mask = np.isin(group_ids, list(validation_groups))
        val_idx = np.flatnonzero(val_mask)
        train_pool_idx = np.flatnonzero(~val_mask)
        train_idx = train_pool_idx[y_model[train_pool_idx] >= 0]
        folds.append((train_idx, val_idx))
    return folds


folds = make_stratified_group_folds(
    y_evaluation, groups, n_splits=n_splits, seed=CONFIG["random_seed"]
)


def minimum_same_class_time_separation(train_idx, val_idx):
    minima = []
    timestamps = model_metadata["timestamp"].to_numpy()
    for class_id in range(len(class_names)):
        train_times = timestamps[train_idx[y_model[train_idx] == class_id]]
        val_times = timestamps[val_idx[y_model[val_idx] == class_id]]
        if len(train_times) and len(val_times):
            distances = [
                abs((pd.Timestamp(a) - pd.Timestamp(b)).total_seconds())
                for a in train_times for b in val_times
            ]
            minima.append(min(distances))
    return min(minima) if minima else np.nan


fold_audit_rows = []
for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    train_groups = set(groups[train_idx])
    val_groups = set(groups[val_idx])
    train_hashes = set(model_metadata.loc[train_idx, "payload_hash"])
    val_hashes = set(model_metadata.loc[val_idx, "payload_hash"])
    train_sessions = set(model_metadata.loc[train_idx, "session_id"])
    val_sessions = set(model_metadata.loc[val_idx, "session_id"])

    assert train_groups.isdisjoint(val_groups)
    assert train_hashes.isdisjoint(val_hashes)
    assert set(y_model[train_idx]) == set(range(len(class_names)))
    assert set(y_evaluation[val_idx]) == set(evaluation_class_names)
    assert np.all(y_model[train_idx] >= 0)

    fold_audit_rows.append({
        "fold": fold,
        "known_train_pool_records": len(train_idx),
        "validation_records": len(val_idx),
        "validation_unknown_records": int(np.sum(y_evaluation[val_idx] == "UNKNOWN")),
        "train_classes": len(np.unique(y_model[train_idx])),
        "validation_targets": len(np.unique(y_evaluation[val_idx])),
        "group_overlap": len(train_groups & val_groups),
        "payload_hash_overlap": len(train_hashes & val_hashes),
        "train_sessions": len(train_sessions),
        "validation_sessions": len(val_sessions),
        "session_overlap": len(train_sessions & val_sessions),
        "minimum_same_class_time_separation_s": minimum_same_class_time_separation(train_idx, val_idx),
    })

fold_audit = pd.DataFrame(fold_audit_rows)
display(fold_audit)
assert fold_audit["group_overlap"].eq(0).all()
assert fold_audit["payload_hash_overlap"].eq(0).all()


,fold,known_train_pool_records,validation_records,validation_unknown_records,train_classes,validation_targets,group_overlap,payload_hash_overlap,train_sessions,validation_sessions,session_overlap,minimum_same_class_time_separation_s
0,1,119,440,249,2,3,0,0,1,2,1,5.049477
1,2,287,272,249,2,3,0,0,1,2,1,6.899199
2,3,291,268,249,2,3,0,0,1,2,1,5.875030
3,4,292,268,250,2,3,0,0,1,2,1,5.901684
4,5,251,308,249,2,3,0,0,1,2,1,5.049477


In [12]:
# Check the model inputs.
learning_input_audit = pd.DataFrame([
    {
        "input": "raw preamble I/Q",
        "shape_per_record": str(tuple(X_preamble.shape[1:])),
        "original_sample_range": "0-79",
        "value_preprocessing": "stored unchanged; neural channel mean/std fitted on each training fold",
        "payload_samples_present": False,
    },
    {
        "input": "33 physical preamble features",
        "shape_per_record": str(tuple(X_physical.shape[1:])),
        "original_sample_range": "derived from 0-79",
        "value_preprocessing": "SVM columns standardized inside each training fold; RF unchanged",
        "payload_samples_present": False,
    },
])
display(learning_input_audit)

payload_start = CONFIG["preamble_len"]
icao_payload_sample_start = payload_start + 8 * 2 * CONFIG["samples_per_half_symbol"]
icao_payload_sample_stop = payload_start + 32 * 2 * CONFIG["samples_per_half_symbol"]
print("Payload starts at sample", payload_start)
print("ICAO-bearing payload samples:", icao_payload_sample_start, "to", icao_payload_sample_stop - 1)
print("Latest model-input sample:", CONFIG["preamble_len"] - 1)
assert X_preamble.shape[-1] <= payload_start


,input,shape_per_record,original_sample_range,value_preprocessing,payload_samples_present
0,raw preamble I/Q,"(2, 80)",0-79,stored unchanged; neural channel mean/std fitt...,False
1,33 physical preamble features,"(33,)",derived from 0-79,SVM columns standardized inside each training ...,False


Payload starts at sample 80
ICAO-bearing payload samples: 160 to 399
Latest model-input sample: 79


## Models

- The RBF SVM and random forest use 33 physical features calculated from preamble I/Q. The SVM's feature scaling is fitted separately inside each training fold.
- The compact CNN, TCN and bidirectional GRU use the $2	imes80$ I/Q sequence after channel-wise scaling fitted on the current training fold. Their channel/hidden widths are deliberately small for this dataset.
- MobileNetV3-Small (scratch and ImageNet transfer) is removed: the image prior and parameter count are not justified by the sample size or the narrow I/Q geometry.
- In open-set mode, a group-held-out subset of the known-class outer-training data calibrates a confidence threshold. Unknown aircraft never fit the classifier or the threshold.


In [13]:
# Neural model definitions.
class ArrayDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.as_tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index] if self.y is None else (self.X[index], self.y[index])


class PreambleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        c1, c2, c3 = CONFIG["cnn_channels"]
        self.network = nn.Sequential(
            nn.Conv1d(2, c1, 7, padding=3), nn.BatchNorm1d(c1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(c1, c2, 5, padding=2), nn.BatchNorm1d(c2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(c2, c3, 3, padding=1), nn.BatchNorm1d(c3), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(c3, num_classes),
        )

    def forward(self, x):
        return self.network(x)


class TCNBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm1d(channels), nn.ReLU(), nn.Dropout(CONFIG["dropout"] * 0.5),
            nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm1d(channels),
        )
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(x + self.block(x))


class PreambleTCN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        channels = CONFIG["tcn_channels"]
        self.input = nn.Conv1d(2, channels, 1)
        self.blocks = nn.Sequential(*(TCNBlock(channels, d) for d in (1, 2, 4)))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(channels, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.blocks(self.input(x)))


class PreambleGRU(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.gru = nn.GRU(2, CONFIG["gru_hidden"], batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.LayerNorm(2 * CONFIG["gru_hidden"]), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(2 * CONFIG["gru_hidden"], num_classes),
        )

    def forward(self, x):
        _, hidden = self.gru(x.transpose(1, 2))
        return self.classifier(torch.cat([hidden[-2], hidden[-1]], dim=1))


MODEL_NAMES = {
    "physical_feature_svm": "Standardized physical-feature RBF SVM",
    "physical_feature_random_forest": "Physical-feature random forest",
    "preamble_cnn": "Compact 1D preamble CNN",
    "preamble_tcn": "Compact temporal convolutional network",
    "preamble_gru_rnn": "Compact bidirectional GRU",
}

def torch_model_spec(experiment, model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "experiment": experiment,
        "model": MODEL_NAMES[experiment],
        "input": "2 x 80 I/Q; fold-local channel scaling",
        "weight_source": "outer training fold",
        "total_parameters": total,
        "trainable_parameters": trainable,
        "parameter_memory_mb": total * 4 / 1e6,
    }

spec_models = {
    "preamble_cnn": PreambleCNN(len(class_names)),
    "preamble_tcn": PreambleTCN(len(class_names)),
    "preamble_gru_rnn": PreambleGRU(len(class_names)),
}
model_specs_df = pd.DataFrame([
    torch_model_spec(name, model) for name, model in spec_models.items()
])
display(model_specs_df)
del spec_models


,experiment,model,input,weight_source,total_parameters,trainable_parameters,parameter_memory_mb
0,preamble_cnn,Compact 1D preamble CNN,2 x 80 I/Q; fold-local channel scaling,outer training fold,7778,7778,0.031112
1,preamble_tcn,Compact temporal convolutional network,2 x 80 I/Q; fold-local channel scaling,outer training fold,10922,10922,0.043688
2,preamble_gru_rnn,Compact bidirectional GRU,2 x 80 I/Q; fold-local channel scaling,outer training fold,7170,7170,0.028680


In [14]:
# Training, calibration and scoring helpers.
def make_loader(X, labels=None, shuffle=False):
    return DataLoader(
        ArrayDataset(X, labels),
        batch_size=CONFIG["batch_size"],
        shuffle=shuffle,
        num_workers=0,
    )


def class_weights(train_labels, num_classes):
    counts = np.bincount(train_labels, minlength=num_classes).astype(np.float32)
    weights = counts.sum() / (num_classes * np.maximum(counts, 1.0))
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def train_supervised(model, X_train, y_train, seed, epochs, learning_rate):
    set_seed(seed)
    model = model.to(DEVICE)
    loader = make_loader(X_train, y_train, shuffle=True)
    criterion = nn.CrossEntropyLoss(weight=class_weights(y_train, len(class_names)))
    optimiser = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=CONFIG["weight_decay"]
    )
    history = []
    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimiser.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimiser.step()
            total_loss += loss.item() * len(yb)
        history.append(total_loss / len(loader.dataset))
    return model, history


@torch.no_grad()
def predict_neural_probabilities(model, X):
    model.eval()
    outputs = []
    for xb in make_loader(X):
        outputs.append(torch.softmax(model(xb.to(DEVICE)), dim=1).cpu().numpy())
    return np.concatenate(outputs)


def split_fit_calibration(train_idx, seed):
    if not IS_OPEN_SET:
        return np.asarray(train_idx), np.array([], dtype=int)

    rng = np.random.default_rng(seed)
    calibration_groups = set()
    for class_id in range(len(class_names)):
        class_idx = train_idx[y_model[train_idx] == class_id]
        class_groups, group_counts = np.unique(groups[class_idx], return_counts=True)
        if len(class_groups) < 2:
            raise ValueError("Open-set calibration needs at least two training groups per known class")

        # Select whole groups whose record total is as close as possible to the
        # requested fraction. Dynamic programming is exact and cheap here.
        order = rng.permutation(len(class_groups))
        class_groups = class_groups[order]
        group_counts = group_counts[order]
        possible = {0: ()}
        for group_id, group_count in zip(class_groups, group_counts):
            additions = {
                total + int(group_count): selected + (group_id,)
                for total, selected in list(possible.items())
                if total + int(group_count) < len(class_idx)
            }
            for total, selected in additions.items():
                possible.setdefault(total, selected)

        target_records = CONFIG["calibration_fraction"] * len(class_idx)
        selected_total = min(
            (total for total in possible if total > 0),
            key=lambda total: (abs(total - target_records), total),
        )
        calibration_groups.update(possible[selected_total])

    calibration_mask = np.isin(groups[train_idx], list(calibration_groups))
    fit_idx = train_idx[~calibration_mask]
    calibration_idx = train_idx[calibration_mask]
    assert set(y_model[fit_idx]) == set(range(len(class_names)))
    assert set(y_model[calibration_idx]) == set(range(len(class_names)))
    assert set(groups[fit_idx]).isdisjoint(set(groups[calibration_idx]))
    return fit_idx, calibration_idx


def calibrated_unknown_threshold(calibration_probabilities):
    if not IS_OPEN_SET:
        return np.nan
    confidences = calibration_probabilities.max(axis=1)
    rejection_quantile = 1.0 - CONFIG["known_acceptance_target"]
    return float(np.quantile(confidences, rejection_quantile, method="lower"))


def labels_from_probabilities(probabilities, threshold):
    predicted_indices = np.argmax(probabilities, axis=1)
    predicted_labels = np.asarray(class_names, dtype=object)[predicted_indices]
    if IS_OPEN_SET:
        predicted_labels = predicted_labels.astype(object)
        predicted_labels[probabilities.max(axis=1) < threshold] = "UNKNOWN"
    return predicted_labels


def score_predictions(targets, predictions):
    scores = {
        "accuracy": accuracy_score(targets, predictions),
        "balanced_accuracy": balanced_accuracy_score(targets, predictions),
        "macro_f1": f1_score(
            targets, predictions, labels=evaluation_class_names, average="macro", zero_division=0
        ),
        "known_accuracy": accuracy_score(
            np.asarray(targets)[np.asarray(targets) != "UNKNOWN"],
            np.asarray(predictions)[np.asarray(targets) != "UNKNOWN"],
        ),
        "unknown_recall": np.nan,
    }
    if "UNKNOWN" in evaluation_class_names:
        scores["unknown_recall"] = recall_score(
            targets, predictions, labels=["UNKNOWN"], average="macro", zero_division=0
        )
    return scores


def torch_complexity(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, total * 4 / 1e6


In [15]:
# Run every model on the same outer folds.
experiment_rows = []
prediction_rows = []
complexity_rows = []
training_histories = {}

for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):
    fit_idx, calibration_idx = split_fit_calibration(
        train_idx, CONFIG["random_seed"] + fold_number
    )
    print(
        f"Fold {fold_number}/{n_splits}: fit={len(fit_idx)}, "
        f"calibration={len(calibration_idx)}, validation={len(val_idx)}"
    )

    for experiment_number, experiment in enumerate(CONFIG["experiments"]):
        seed = CONFIG["random_seed"] + 100 * fold_number + experiment_number
        set_seed(seed)
        history = None

        if experiment == "physical_feature_svm":
            model = make_pipeline(
                StandardScaler(),
                SVC(
                    C=3.0, kernel="rbf", gamma="scale", class_weight="balanced",
                    probability=True, random_state=seed,
                ),
            )
            model.fit(X_physical[fit_idx], y_model[fit_idx])
            validation_probabilities = model.predict_proba(X_physical[val_idx])
            calibration_probabilities = (
                model.predict_proba(X_physical[calibration_idx]) if len(calibration_idx) else None
            )
            fitted_svc = model.named_steps["svc"]
            complexity_rows.append({
                "experiment": experiment,
                "fold": fold_number,
                "total_parameters": np.nan,
                "trainable_parameters": np.nan,
                "parameter_memory_mb": np.nan,
                "support_vectors": fitted_svc.support_vectors_.shape[0],
                "tree_nodes": np.nan,
                "fitted_size_mb": len(pickle.dumps(model)) / 1e6,
            })

        elif experiment == "physical_feature_random_forest":
            model = RandomForestClassifier(
                n_estimators=CONFIG["random_forest_trees"],
                min_samples_leaf=2,
                max_features="sqrt",
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=seed,
            )
            model.fit(X_physical[fit_idx], y_model[fit_idx])
            validation_probabilities = model.predict_proba(X_physical[val_idx])
            calibration_probabilities = (
                model.predict_proba(X_physical[calibration_idx]) if len(calibration_idx) else None
            )
            complexity_rows.append({
                "experiment": experiment,
                "fold": fold_number,
                "total_parameters": np.nan,
                "trainable_parameters": np.nan,
                "parameter_memory_mb": np.nan,
                "support_vectors": np.nan,
                "tree_nodes": sum(tree.tree_.node_count for tree in model.estimators_),
                "fitted_size_mb": len(pickle.dumps(model)) / 1e6,
            })

        else:
            if experiment == "preamble_cnn":
                model = PreambleCNN(len(class_names))
            elif experiment == "preamble_tcn":
                model = PreambleTCN(len(class_names))
            elif experiment == "preamble_gru_rnn":
                model = PreambleGRU(len(class_names))
            else:
                raise ValueError(experiment)

            channel_mean = X_preamble[fit_idx].mean(axis=(0, 2), keepdims=True)
            channel_std = X_preamble[fit_idx].std(axis=(0, 2), keepdims=True) + 1e-6
            X_data = (X_preamble - channel_mean) / channel_std

            model, history = train_supervised(
                model, X_data[fit_idx], y_model[fit_idx], seed,
                CONFIG["epochs"], CONFIG["learning_rate"],
            )
            validation_probabilities = predict_neural_probabilities(model, X_data[val_idx])
            calibration_probabilities = (
                predict_neural_probabilities(model, X_data[calibration_idx])
                if len(calibration_idx) else None
            )
            total, trainable, memory_mb = torch_complexity(model)
            complexity_rows.append({
                "experiment": experiment,
                "fold": fold_number,
                "total_parameters": total,
                "trainable_parameters": trainable,
                "parameter_memory_mb": memory_mb,
                "support_vectors": np.nan,
                "tree_nodes": np.nan,
                "fitted_size_mb": memory_mb,
            })
            training_histories[(experiment, fold_number)] = history

        threshold = (
            calibrated_unknown_threshold(calibration_probabilities)
            if calibration_probabilities is not None else np.nan
        )
        predictions = labels_from_probabilities(validation_probabilities, threshold)
        targets = y_evaluation[val_idx]
        metrics = score_predictions(targets, predictions)
        experiment_rows.append({
            "experiment": experiment,
            "fold": fold_number,
            "validation_kind": evidence_grade,
            "n_fit": len(fit_idx),
            "n_calibration": len(calibration_idx),
            "n_validation": len(val_idx),
            "unknown_threshold": threshold,
            **metrics,
        })

        for row_index, target, prediction, confidence in zip(
            val_idx, targets, predictions, validation_probabilities.max(axis=1)
        ):
            prediction_rows.append({
                "experiment": experiment,
                "fold": fold_number,
                "record_index": int(model_metadata.loc[row_index, "record_index"]),
                "capture_file": model_metadata.loc[row_index, "capture_file"],
                "source_record_index": int(model_metadata.loc[row_index, "source_record_index"]),
                "true_icao": model_metadata.loc[row_index, "icao"],
                "true_class": target,
                "predicted_class": prediction,
                "maximum_known_probability": float(confidence),
                "unknown_threshold": threshold,
            })

        detail = f" unknown-recall={metrics['unknown_recall']:.3f}" if IS_OPEN_SET else ""
        print(
            f"  {MODEL_NAMES[experiment]:43s} balanced={metrics['balanced_accuracy']:.3f} "
            f"macro-F1={metrics['macro_f1']:.3f}{detail}"
        )

results_df = pd.DataFrame(experiment_rows)
predictions_df = pd.DataFrame(prediction_rows)
complexity_df = pd.DataFrame(complexity_rows)
display(results_df)


Fold 1/5: fit=95, calibration=24, validation=440
  Standardized physical-feature RBF SVM       balanced=0.336 macro-F1=0.216 unknown-recall=0.100


  Physical-feature random forest              balanced=0.251 macro-F1=0.214 unknown-recall=0.112


  Compact 1D preamble CNN                     balanced=0.294 macro-F1=0.249 unknown-recall=0.177


  Compact temporal convolutional network      balanced=0.310 macro-F1=0.246 unknown-recall=0.124


  Compact bidirectional GRU                   balanced=0.217 macro-F1=0.179 unknown-recall=0.016
Fold 2/5: fit=232, calibration=55, validation=272
  Standardized physical-feature RBF SVM       balanced=0.366 macro-F1=0.132 unknown-recall=0.165


...

  Physical-feature random forest              balanced=0.331 macro-F1=0.068 unknown-recall=0.064


  Compact 1D preamble CNN                     balanced=0.355 macro-F1=0.071 unknown-recall=0.044


  Compact temporal convolutional network      balanced=0.370 macro-F1=0.144 unknown-recall=0.044


  Compact bidirectional GRU                   balanced=0.366 macro-F1=0.126 unknown-recall=0.016


,experiment,fold,validation_kind,n_fit,n_calibration,n_validation,unknown_threshold,accuracy,balanced_accuracy,macro_f1,known_accuracy,unknown_recall
0,physical_feature_svm,1,session-aware temporal-burst grouped evaluation,95,24,440,0.529425,0.234091,0.335606,0.215649,0.408377,0.100402
1,physical_feature_random_forest,1,session-aware temporal-burst grouped evaluation,95,24,440,0.550691,0.231818,0.250941,0.213638,0.387435,0.112450
2,preamble_cnn,1,session-aware temporal-burst grouped evaluation,95,24,440,0.525476,0.295455,0.293751,0.249414,0.450262,0.176707
3,preamble_tcn,1,session-aware temporal-burst grouped evaluation,95,24,440,0.542431,0.309091,0.310216,0.246198,0.549738,0.124498
4,preamble_gru_rnn,1,session-aware temporal-burst grouped evaluation,95,24,440,0.503788,0.279545,0.217476,0.179371,0.623037,0.016064
5,physical_feature_svm,2,session-aware temporal-burst grouped evaluation,232,55,272,0.778201,0.202206,0.365997,0.131859,0.608696,0.164659
6,physical_feature_random_forest,2,session-aware temporal-burst grouped evaluation,232,55,272,0.565700,0.136029,0.341901,0.093853,0.608696,0.092369
7,preamble_cnn,2,session-aware temporal-burst grouped evaluation,232,55,272,0.525098,0.161765,0.367838,0.134603,0.521739,0.128514
8,preamble_tcn,2,session-aware temporal-burst grouped evaluation,232,55,272,0.508598,0.077206,0.337048,0.080474,0.521739,0.036145
9,preamble_gru_rnn,2,session-aware temporal-burst grouped evaluation,232,55,272,0.509157,0.069853,0.313487,0.069792,0.478261,0.032129


In [16]:
# Combined out-of-fold results.
overall_rows = []
for experiment in CONFIG["experiments"]:
    subset = predictions_df[predictions_df["experiment"] == experiment]
    overall_rows.append({
        "experiment": experiment,
        "model": MODEL_NAMES[experiment],
        **score_predictions(subset["true_class"], subset["predicted_class"]),
        "mean_unknown_threshold": results_df.loc[
            results_df["experiment"] == experiment, "unknown_threshold"
        ].mean(),
    })
overall_summary = pd.DataFrame(overall_rows).sort_values(
    "balanced_accuracy", ascending=False
).reset_index(drop=True)
display(overall_summary)

uniform_chance = 1.0 / len(evaluation_class_names)
majority_accuracy = model_metadata["evaluation_label"].value_counts(normalize=True).max()
print(f"Balanced chance reference: {uniform_chance:.3f}")
print(f"Majority-target accuracy reference: {majority_accuracy:.3f}")
print("Validation:", evidence_grade)

fig, axis = plt.subplots(figsize=(10, 5))
plot_data = overall_summary.sort_values("balanced_accuracy")
y_pos = np.arange(len(plot_data))
axis.barh(y_pos - 0.18, plot_data["balanced_accuracy"], height=0.36, label="Balanced accuracy")
axis.barh(y_pos + 0.18, plot_data["macro_f1"], height=0.36, label="Macro F1")
axis.axvline(uniform_chance, color="black", linestyle="--", linewidth=1, label="Balanced chance")
axis.set_yticks(y_pos, plot_data["model"])
axis.set_xlim(0, max(0.5, 1.1 * plot_data[["balanced_accuracy", "macro_f1"]].to_numpy().max()))
axis.set_xlabel("Out-of-fold score")
axis.legend()
axis.set_title("Multi-session raw-preamble model comparison")
plt.tight_layout()
fig.savefig(figure_dir / "model_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

short_names = {
    "physical_feature_svm": "RBF SVM",
    "physical_feature_random_forest": "Random forest",
    "preamble_cnn": "1D CNN",
    "preamble_tcn": "TCN",
    "preamble_gru_rnn": "Bidirectional GRU",
}
n_cols = 3
n_rows = int(np.ceil(len(CONFIG["experiments"]) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows), squeeze=False)
for axis, experiment in zip(axes.ravel(), CONFIG["experiments"]):
    subset = predictions_df[predictions_df["experiment"] == experiment]
    cm = confusion_matrix(
        subset["true_class"], subset["predicted_class"],
        labels=evaluation_class_names, normalize="true",
    )
    image = axis.imshow(cm, vmin=0, vmax=1, aspect="auto")
    axis.set_title(short_names[experiment], fontsize=10)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")
    if len(evaluation_class_names) <= 10:
        axis.set_xticks(range(len(evaluation_class_names)), evaluation_class_names, rotation=45, ha="right")
        axis.set_yticks(range(len(evaluation_class_names)), evaluation_class_names)
    else:
        axis.set_xticks([])
        axis.set_yticks([])

    single_fig, single_axis = plt.subplots(figsize=(7, 6))
    single_image = single_axis.imshow(cm, vmin=0, vmax=1, aspect="auto")
    single_axis.set_title(MODEL_NAMES[experiment])
    single_axis.set_xlabel("Predicted class")
    single_axis.set_ylabel("True class")
    if len(evaluation_class_names) <= 10:
        single_axis.set_xticks(
            range(len(evaluation_class_names)), evaluation_class_names, rotation=45, ha="right"
        )
        single_axis.set_yticks(range(len(evaluation_class_names)), evaluation_class_names)
    else:
        single_axis.set_xticks([])
        single_axis.set_yticks([])
    single_fig.colorbar(single_image, ax=single_axis, label="Per-class fraction")
    single_fig.tight_layout()
    single_fig.savefig(figure_dir / f"{experiment}_confusion.png", dpi=180, bbox_inches="tight")
    plt.close(single_fig)

for axis in axes.ravel()[len(CONFIG["experiments"]):]:
    axis.axis("off")
fig.colorbar(image, ax=axes.ravel().tolist(), label="Per-class fraction", shrink=0.8)
fig.suptitle("Multi-session specific-transmitter identification")
fig.savefig(figure_dir / "confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.show()

complexity_summary = (
    complexity_df.drop(columns="fold")
    .groupby("experiment", as_index=False)
    .mean(numeric_only=True)
)
model_specs_complete = pd.DataFrame([
    {
        "experiment": experiment,
        "model": MODEL_NAMES[experiment],
        "input": "33 physical preamble features" if "physical_feature" in experiment else "2 x 80 I/Q; fold-local channel scaling",
        "architecture": (
            "RBF SVM with fold-local StandardScaler" if experiment == "physical_feature_svm"
            else f"{CONFIG['random_forest_trees']} trees, minimum leaf 2" if experiment == "physical_feature_random_forest"
            else f"3 convolution blocks: {CONFIG['cnn_channels']} channels" if experiment == "preamble_cnn"
            else f"3 residual dilated blocks, {CONFIG['tcn_channels']} channels" if experiment == "preamble_tcn"
            else f"1 bidirectional GRU layer, {CONFIG['gru_hidden']} units per direction"
        ),
        "weight_source": "outer training fold",
    }
    for experiment in CONFIG["experiments"]
]).merge(complexity_summary, on="experiment", how="left")
display(model_specs_complete)


,experiment,model,accuracy,balanced_accuracy,macro_f1,known_accuracy,unknown_recall,mean_unknown_threshold
0,preamble_gru_rnn,Compact bidirectional GRU,0.138175,0.374303,0.133583,0.596774,0.024077,0.504221
1,preamble_tcn,Compact temporal convolutional network,0.161311,0.351026,0.150228,0.567742,0.060193,0.521654
2,preamble_cnn,Compact 1D preamble CNN,0.161954,0.246047,0.136324,0.461290,0.087480,0.515641
3,physical_feature_svm,Standardized physical-feature RBF SVM,0.150386,0.217118,0.114547,0.435484,0.079454,0.688572
4,physical_feature_random_forest,Physical-feature random forest,0.138817,0.204100,0.105793,0.422581,0.068218,0.597065


Balanced chance reference: 0.333
Majority-target accuracy reference: 0.801
Validation: session-aware temporal-burst grouped evaluation


/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_2112/3451992434.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_2112/3451992434.py:89: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,experiment,model,input,architecture,weight_source,total_parameters,trainable_parameters,parameter_memory_mb,support_vectors,tree_nodes,fitted_size_mb
0,physical_feature_svm,Standardized physical-feature RBF SVM,33 physical preamble features,RBF SVM with fold-local StandardScaler,outer training fold,NaN,NaN,NaN,140.0,NaN,0.042180
1,physical_feature_random_forest,Physical-feature random forest,33 physical preamble features,"200 trees, minimum leaf 2",outer training fold,NaN,NaN,NaN,NaN,8392.8,0.733685
2,preamble_cnn,Compact 1D preamble CNN,2 x 80 I/Q; fold-local channel scaling,"3 convolution blocks: (16, 32, 48) channels",outer training fold,7778.0,7778.0,0.031112,NaN,NaN,0.031112
3,preamble_tcn,Compact temporal convolutional network,2 x 80 I/Q; fold-local channel scaling,"3 residual dilated blocks, 24 channels",outer training fold,10922.0,10922.0,0.043688,NaN,NaN,0.043688
4,preamble_gru_rnn,Compact bidirectional GRU,2 x 80 I/Q; fold-local channel scaling,"1 bidirectional GRU layer, 32 units per direction",outer training fold,7170.0,7170.0,0.028680,NaN,NaN,0.028680


In [17]:
# Save results and audit files.
output_dir = CONFIG["artifact_dir"]
results_df.to_csv(output_dir / "grouped_fold_results.csv", index=False)
overall_summary.to_csv(output_dir / "aggregate_results.csv", index=False)
predictions_df.to_csv(output_dir / "out_of_fold_predictions.csv", index=False)
fold_audit.to_csv(output_dir / "fold_leakage_audit.csv", index=False)
learning_input_audit.to_csv(output_dir / "model_input_audit.csv", index=False)
model_specs_complete.to_csv(output_dir / "model_specs.csv", index=False)
capture_file_audit.to_csv(output_dir / "capture_file_audit.csv", index=False)
class_session_counts.to_csv(output_dir / "class_session_counts.csv")

assignments = model_metadata[[
    "record_index", "source_record_index", "capture_file", "session_id", "timestamp",
    "icao", "evaluation_label", "is_known_class", "group_id",
]].copy()
assignments["validation_fold"] = -1
for fold_number, (_, val_idx) in enumerate(folds, start=1):
    assignments.loc[val_idx, "validation_fold"] = fold_number
assert assignments["validation_fold"].gt(0).all()
assignments.to_csv(output_dir / "fold_assignments.csv", index=False)

with (output_dir / "label_mapping.json").open("w") as stream:
    json.dump({int(index): label for index, label in enumerate(class_names)}, stream, indent=2)

serialisable_config = {
    key: (str(value) if isinstance(value, Path) else value) for key, value in CONFIG.items()
}
with (output_dir / "experiment_config.json").open("w") as stream:
    json.dump(serialisable_config, stream, indent=2)

print("Saved:", output_dir.resolve())


Saved: /Users/calebrodgers/Desktop/ml/adsb_specific_transmitter_two_class_outputs


## Interpretation guardrails

- Balanced accuracy and macro F1 are primary because class counts differ substantially.
- Above-chance performance demonstrates class-correlated preamble information, not yet a session-invariant hardware fingerprint.
- A genuine session/day generalisation claim needs the same evaluated aircraft in several independently collected sessions so complete sessions can be held out.
- The two-class `UNKNOWN` decision is confidence rejection, not a learned third class: unknown-aircraft samples are evaluation-only.
- Raw I/Q retains absolute amplitude, phase, receiver, range, Doppler and channel effects; those can remain shortcuts until a stronger multi-session design is available.
